# DoH-Shield | Phase 1 & 2: Complete End-to-End Training
## Website Fingerprinting Attack Replication & Cluster-Based Privacy Defenses

**Project**: DoH-Shield | CS362IA Network Programming and Security  
**Goal**: 
1. Programmatically download CIRA-CIC-DoHBrw-2020 Parquet dataset using Colab secrets.
2. Preprocess and replicate baseline Website Fingerprinting attacks (Random Forest & Deep Fingerprinting 1D CNN).
3. Train KMeans clustering model (K=30) for DoH-Shield's traffic morphing defense.
4. Compute formal privacy guarantees (l-diversity & differential privacy bound).
5. Zip all generated models, scalers, reports, and plots for local retrieval or Hugging Face Hub upload.

In [ ]:
# CELL 1: Environment Setup & Library Imports
# =============================================================================

# Install necessary packages for processing, ML, DL, and HF uploads
!pip install -q pandas numpy scikit-learn matplotlib seaborn joblib torch pyarrow fastparquet huggingface_hub

import os
import sys
import glob
import math
import time
import zipfile
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter, defaultdict
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (
    classification_report, f1_score, accuracy_score,
    confusion_matrix, roc_auc_score, roc_curve
)
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

# Confirm GPU for PyTorch CNN
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    print(f'✅ GPU detected: {torch.cuda.get_device_name(0)}')
else:
    print('⚠️ Running on CPU. Training the Deep Fingerprinting CNN will be slow!')

print(f'Python Version: {sys.version.split()[0]}')
print(f'PyTorch Version: {torch.__version__}')

---
## CELL 2: Dataset Download

This cell retrieves Kaggle API credentials (`KAGGLE_USERNAME` and `KAGGLE_KEY`) from Google Colab Secrets (the key icon in the left sidebar) to programmatically download the CIRA-CIC-DoHBrw-2020 dataset.

In [ ]:
# CELL 2: Retrieve Credentials & Download Dataset
# =============================================================================
from google.colab import userdata

try:
    # Retrieve credentials from Google Colab Secrets
    kaggle_username = userdata.get('KAGGLE_USERNAME')
    kaggle_key = userdata.get('KAGGLE_KEY')
    
    if kaggle_username and kaggle_key:
        os.environ['KAGGLE_USERNAME'] = kaggle_username
        os.environ['KAGGLE_KEY'] = kaggle_key
        print("✅ Kaggle credentials loaded successfully from Colab Secrets.")
        
        # Download the dataset
        print("📥 Downloading CIRA-CIC-DoHBrw-2020 dataset from Kaggle...")
        !mkdir -p /content/data
        !kaggle datasets download -d dhoogla/cicdohbrw2020 --unzip -p /content/data/ -q
        print("✅ Dataset downloaded and extracted successfully!")
    else:
        raise ValueError("Empty credentials returned.")
except Exception as e:
    print(f"⚠️ Error retrieving credentials: {e}")
    print("Please ensure you have added KAGGLE_USERNAME and KAGGLE_KEY to Colab Secrets (left sidebar -> key icon).")
    print("Fallback: You can manually upload the dataset files to the `/content/data/` folder.")

# Inspect directory
data_dir = '/content/data/'
if os.path.exists(data_dir):
    files_found = glob.glob(data_dir + '**/*', recursive=True)
    print(f"\n📁 Files in /content/data:")
    for f in files_found:
        if os.path.isfile(f):
            print(f"  - {os.path.basename(f)} ({os.path.getsize(f) / (1024*1024):.2f} MB)")

---
## CELL 3: Data Loading & Initial Inspection

We read the Layer 2 parquet file `L2-BenignDoH-MaliciousDoH.parquet` which contains the flow statistical features for benign and malicious DNS-over-HTTPS queries.

In [ ]:
# CELL 3: Load Layer 2 Data (Benign vs Malicious DoH)
# =============================================================================

parquet_files = glob.glob(data_dir + '**/*.parquet', recursive=True)
l2_file = None

for f in parquet_files:
    if 'L2' in os.path.basename(f) or 'l2' in f.lower():
        l2_file = f
        break

if not l2_file and parquet_files:
    l2_file = parquet_files[0]

if l2_file:
    print(f"📖 Loading Layer 2 dataset from: {l2_file}")
    df_raw = pd.read_parquet(l2_file)
    print(f"✅ Loaded successfully! Shape: {df_raw.shape}")
else:
    # Fallback to CSV loading if parquet files aren't found
    csv_files = glob.glob(data_dir + '**/*.csv', recursive=True)
    print(f"📖 No parquet files found. Loading CSV files: {csv_files}")
    df_raw = pd.concat([pd.read_csv(f, low_memory=False) for f in csv_files], ignore_index=True)
    print(f"✅ Loaded successfully from CSV! Shape: {df_raw.shape}")

# Inspect column candidates
label_col = None
for c in ['label', 'Label', 'class', 'type', 'Type']:
    if c in df_raw.columns:
        label_col = c
        break

print(f"Target Label Column: {label_col}")
print("Class Counts:")
print(df_raw[label_col].value_counts())

---
## CELL 4: Data Preprocessing & Scaling

We drop non-informative metadata features (IP addresses and ports, which are not available to a network-level passive eavesdropper) and handle missing/infinite values before performing train/test split.

In [ ]:
# CELL 4: Data Preprocessing
# =============================================================================

df = df_raw.copy()

# Drop network-level identifier columns that are easily changed or leak identity
drop_candidates = [
    'SourceIP', 'Source IP', 'DestinationIP', 'Destination IP',
    'SourcePort', 'Source Port', 'DestinationPort', 'Destination Port',
    'Unnamed: 0', 'index'
]
df.drop(columns=[c for c in drop_candidates if c in df.columns], inplace=True)

# Separate features and label
X = df.drop(columns=[label_col])
y_raw = df[label_col]

# Label encoding: 0 = Benign, 1 = Malicious
le = LabelEncoder()
y = le.fit_transform(y_raw)
print(f"Label encoding: {dict(zip(le.classes_, le.transform(le.classes_)))}")

# Clean feature values
X = X.apply(pd.to_numeric, errors='coerce')

# Drop entirely NaN columns
all_nan_cols = X.columns[X.isna().all()]
X.drop(columns=all_nan_cols, inplace=True)

# Impute missing/infinite values using median
X.fillna(X.median(), inplace=True)
X.replace([np.inf, -np.inf], np.nan, inplace=True)
X.fillna(X.median(), inplace=True)

feature_names = X.columns.tolist()
print(f"Features Count: {len(feature_names)}")

# Train/Test Stratified Split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X.values, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Standard Scale features (fitted ONLY on training data to avoid leakage)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"✅ Preprocessing finished.")
print(f"  - Train Shape: {X_train_scaled.shape}")
print(f"  - Test Shape: {X_test_scaled.shape}")

# Save Scaler and Label Encoder
joblib.dump(scaler, 'feature_scaler.pkl')
joblib.dump(le, 'label_encoder.pkl')
np.save('feature_names.npy', np.array(feature_names))
print("💾 Preprocessing scalers saved locally.")

---
## CELL 5: Exploratory Data Analysis (EDA)

Visualise feature distributions and correlation heatmaps to gain structural intuition on the traffic characteristics.

In [ ]:
# CELL 5: EDA and Visualizations
# =============================================================================

key_features = [
    'Duration', 'FlowBytesSent', 'FlowBytesReceived',
    'PacketLengthMean', 'PacketTimeMean', 'ResponseTimeTimeMean'
]
key_features = [f for f in key_features if f in X.columns]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Feature Distributions: Benign-DoH vs Malicious-DoH', fontsize=14, fontweight='bold')

for ax, feat in zip(axes.flatten(), key_features):
    benign_vals = df[df[label_col].str.lower().str.contains('benign')][feat].dropna()
    malicious_vals = df[~df[label_col].str.lower().str.contains('benign')][feat].dropna()
    
    # Clip to 99th percentile for visibility
    clip_val = np.percentile(pd.concat([benign_vals, malicious_vals]), 99)
    benign_vals = benign_vals.clip(upper=clip_val)
    malicious_vals = malicious_vals.clip(upper=clip_val)
    
    ax.hist(benign_vals, bins=50, alpha=0.6, color='steelblue', label='Benign-DoH', density=True)
    ax.hist(malicious_vals, bins=50, alpha=0.6, color='crimson', label='Malicious-DoH', density=True)
    ax.set_title(feat, fontsize=10)
    ax.legend(fontsize=8)
    ax.set_ylabel('Density')

plt.tight_layout()
plt.savefig('eda_feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

# Feature Correlation Heatmap
plt.figure(figsize=(14, 10))
corr_matrix = X.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, cmap='RdBu_r', center=0,
    vmin=-1, vmax=1, linewidths=0.3, annot=False
)
plt.title('Feature Correlation Matrix (CIRA-CIC-DoHBrw-2020)', fontsize=13)
plt.tight_layout()
plt.savefig('eda_correlation_heatmap.png', dpi=150)
plt.show()

---
## CELL 6: Attack Model 1 — Random Forest Classifier

We train a baseline Random Forest Classifier, evaluate its classification metrics, analyze feature importance, and save the model file.

In [ ]:
# CELL 6: Random Forest Training & Evaluation
# =============================================================================

print("🌲 Training Random Forest Attack Classifier...")
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    min_samples_leaf=2,
    class_weight='balanced',
    n_jobs=-1,
    random_state=42
)

t0 = time.time()
rf.fit(X_train_scaled, y_train)
train_time = time.time() - t0

y_pred_rf = rf.predict(X_test_scaled)
y_prob_rf = rf.predict_proba(X_test_scaled)[:, 1]

rf_accuracy = accuracy_score(y_test, y_pred_rf)
rf_f1 = f1_score(y_test, y_pred_rf, average='weighted')
rf_auc = roc_auc_score(y_test, y_prob_rf)

print(f'\n=== RANDOM FOREST ATTACK RESULTS ===')
print(f'Training time: {train_time:.1f}s')
print(f'Accuracy: {rf_accuracy*100:.2f}%')
print(f'F1 Score: {rf_f1:.4f}')
print(f'ROC-AUC: {rf_auc:.4f}')
print(f'\nDetailed report:')
print(classification_report(y_test, y_pred_rf, target_names=le.classes_))

# Feature Importance Analysis
importances = rf.feature_importances_
sorted_idx = np.argsort(importances)[::-1]
top_features_names = [feature_names[i] for i in sorted_idx[:10]]

plt.figure(figsize=(12, 6))
plt.bar(range(len(feature_names)), importances[sorted_idx], color='steelblue', alpha=0.85)
plt.xticks(range(len(feature_names)), [feature_names[i] for i in sorted_idx], rotation=90, fontsize=8)
plt.title('Random Forest Feature Importance (Undefended Baseline)', fontsize=12)
plt.ylabel('Importance Score')
plt.tight_layout()
plt.savefig('rf_feature_importance.png', dpi=150)
plt.show()

print('Top 10 features representing fingerprint leakage:')
for rank, idx in enumerate(sorted_idx[:10], 1):
    print(f'  {rank:2d}. {feature_names[idx]:<45s} importance={importances[idx]:.4f}')

# Save model and top feature priorities
joblib.dump(rf, 'rf_attack_model.pkl')
np.save('top_features_phase3.npy', np.array(top_features_names))
print("💾 Random Forest model saved locally as `rf_attack_model.pkl`.")

# Plot Confusion Matrix & ROC Curve
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cm = confusion_matrix(y_test, y_pred_rf)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=le.classes_, yticklabels=le.classes_, ax=axes[0])
axes[0].set_title('Random Forest Confusion Matrix', fontsize=12)
axes[0].set_ylabel('True Label')
axes[0].set_xlabel('Predicted Label')

fpr, tpr, _ = roc_curve(y_test, y_prob_rf)
axes[1].plot(fpr, tpr, color='crimson', lw=2, label=f'RF (AUC = {rf_auc:.3f})')
axes[1].plot([0, 1], [0, 1], 'k--', lw=1, label='Random classifier')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve — RF Attacker', fontsize=12)
axes[1].legend(loc='lower right')
plt.tight_layout()
plt.savefig('rf_attack_results.png', dpi=150)
plt.show()

---
## CELL 7: Attack Model 2 — Deep Fingerprinting CNN

We define a robust 1D Convolutional Neural Network modeled after the *Deep Fingerprinting* architecture (Sirinam et al., CCS 2018), compile it with PyTorch, train it with Cosine Annealing, and save the best weight state.

In [ ]:
# CELL 7: CNN Attack Classifier Model (Deep Fingerprinting)
# =============================================================================

class DeepFingerprint(nn.Module):
    """
    1D CNN for website fingerprinting.
    Adapted from Sirinam et al. for tabular DoH flow representations.
    """
    def __init__(self, input_dim: int, num_classes: int = 2):
        super(DeepFingerprint, self).__init__()
        
        self.block1 = nn.Sequential(
            nn.Conv1d(in_channels=1, out_channels=32, kernel_size=5, padding=2),
            nn.BatchNorm1d(32),
            nn.ELU(),
            nn.MaxPool1d(kernel_size=2, stride=2),
            nn.Dropout(0.1)
        )
        
        self.block2 = nn.Sequential(
            nn.Conv1d(in_channels=32, out_channels=64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64),
            nn.ELU(),
            nn.MaxPool1d(kernel_size=2, stride=2),
            nn.Dropout(0.1)
        )
        
        # Calculate linear layer dimensions
        conv_out_dim = 64 * (input_dim // 4)
        
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(conv_out_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )
    
    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        return self.classifier(x)

# Tensors conversion - shape [N, 1, features] for 1D CNN
X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32).unsqueeze(1)
X_test_t  = torch.tensor(X_test_scaled,  dtype=torch.float32).unsqueeze(1)
y_train_t = torch.tensor(y_train, dtype=torch.long)
y_test_t  = torch.tensor(y_test,  dtype=torch.long)

# DataLoaders Setup
BATCH_SIZE = 512
EPOCHS = 40
LEARNING_RATE = 1e-3

train_ds = TensorDataset(X_train_t, y_train_t)
test_ds  = TensorDataset(X_test_t,  y_test_t)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# Class Weights to offset dataset imbalance
class_counts = np.bincount(y_train)
class_weights = torch.tensor(len(y_train) / (2.0 * class_counts), dtype=torch.float32).to(device)

input_dim = X_train_scaled.shape[1]
cnn_model = DeepFingerprint(input_dim=input_dim, num_classes=2).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(cnn_model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

print(f"🚀 Training Deep Fingerprinting CNN for {EPOCHS} epochs...")
train_losses, val_losses = [], []
train_accs, val_accs = [], []
best_f1 = 0.0

print(f'{"Epoch":>6} | {"Train Loss":>10} | {"Train Acc":>9} | {"Val Loss":>9} | {"Val Acc":>8} | {"Val F1":>7}')
print('-' * 65)

for epoch in range(1, EPOCHS + 1):
    # Training Pass
    cnn_model.train()
    running_loss, correct, total = 0.0, 0, 0
    for X_b, y_b in train_loader:
        X_b, y_b = X_b.to(device), y_b.to(device)
        optimizer.zero_grad()
        logits = cnn_model(X_b)
        loss = criterion(logits, y_b)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * X_b.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == y_b).sum().item()
        total += X_b.size(0)
        
    epoch_train_loss = running_loss / total
    epoch_train_acc  = correct / total
    
    # Validation Pass
    cnn_model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    all_preds, all_true = [], []
    with torch.no_grad():
        for X_b, y_b in test_loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            logits = cnn_model(X_b)
            loss = criterion(logits, y_b)
            
            val_loss += loss.item() * X_b.size(0)
            preds = logits.argmax(dim=1)
            val_correct += (preds == y_b).sum().item()
            val_total += X_b.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_true.extend(y_b.cpu().numpy())
            
    epoch_val_loss = val_loss / val_total
    epoch_val_acc  = val_correct / val_total
    epoch_val_f1   = f1_score(all_true, all_preds, average='weighted')
    
    train_losses.append(epoch_train_loss)
    val_losses.append(epoch_val_loss)
    train_accs.append(epoch_train_acc)
    val_accs.append(epoch_val_acc)
    
    # Save weights with best validation F1
    if epoch_val_f1 > best_f1:
        best_f1 = epoch_val_f1
        torch.save(cnn_model.state_dict(), 'df_attack_model_best.pt')
        
    scheduler.step()
    
    if epoch % 5 == 0 or epoch == 1:
        print(f'{epoch:>6} | {epoch_train_loss:>10.4f} | {epoch_train_acc:>9.4f} | {epoch_val_loss:>9.4f} | {epoch_val_acc:>8.4f} | {epoch_val_f1:>7.4f}')

print(f"\n✅ CNN Training complete. Best validation F1: {best_f1:.4f}")

# Final CNN Evaluation
cnn_model.load_state_dict(torch.load('df_attack_model_best.pt'))
cnn_model.eval()
all_preds, all_true = [], []
with torch.no_grad():
    for X_b, y_b in test_loader:
        X_b = X_b.to(device)
        logits = cnn_model(X_b)
        preds = logits.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_true.extend(y_b.numpy())

print(classification_report(all_true, all_preds, target_names=le.classes_))

# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(train_losses, label='Train Loss', color='steelblue')
axes[0].plot(val_losses, label='Val Loss', color='crimson')
axes[0].set_title('CNN Loss Curves')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()

axes[1].plot(train_accs, label='Train Acc', color='steelblue')
axes[1].plot(val_accs, label='Val Acc', color='crimson')
axes[1].set_title('CNN Accuracy Curves')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()

plt.tight_layout()
plt.savefig('cnn_training_curves.png', dpi=150)
plt.show()

---
## CELL 8: Phase 2 KMeans Cluster Scaler Setup

For clustering, we fit a fresh scaler on ALL data samples (representing the full demographic flow characteristics) to maximize distance alignment.

In [ ]:
# CELL 8: Cluster Scaling Setup
# =============================================================================

cluster_scaler = StandardScaler()
X_scaled_all = cluster_scaler.fit_transform(X.values)

joblib.dump(cluster_scaler, 'cluster_scaler.pkl')
print("✅ Fit and saved independent cluster_scaler.pkl fitted on all traffic samples.")

---
## CELL 9: PCA Dimensionality Projection

Verify structural separability and density of the flow features in lower dimensions using PCA.

In [ ]:
# CELL 9: PCA Projection plots
# =============================================================================

sample_idx = np.random.choice(len(X_scaled_all), size=20000, replace=False)
X_sample   = X_scaled_all[sample_idx]
y_sample   = y[sample_idx]

pca = PCA(n_components=2, random_state=42)
X_2d = pca.fit_transform(X_sample)
explained = pca.explained_variance_ratio_

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
colors = ['steelblue' if c == 0 else 'crimson' for c in y_sample]
axes[0].scatter(X_2d[:, 0], X_2d[:, 1], c=colors, s=3, alpha=0.4)
axes[0].set_title(f'PCA of DoH Flow Features\n(PC1 vs PC2 | Variance: {sum(explained):.1%})')
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='steelblue', label='Benign-DoH'),
                   Patch(facecolor='crimson', label='Malicious-DoH')]
axes[0].legend(handles=legend_elements)

axes[1].hexbin(X_2d[:, 0], X_2d[:, 1], gridsize=60, cmap='YlOrRd', mincnt=1)
axes[1].set_title('PCA Density Mapping')
axes[1].set_xlabel('PC1')
axes[1].set_ylabel('PC2')

plt.tight_layout()
plt.savefig('pca_cluster_structure.png', dpi=150)
plt.show()

---
## CELL 10: Elbow Analysis for Optimal K

Run rapid elbow iteration using MiniBatchKMeans to discover the mathematically justified cluster size trade-off.

In [ ]:
# CELL 10: MiniBatchKMeans Elbow Plotting
# =============================================================================

K_values  = list(range(5, 85, 5))
inertias  = []

print('🏃 Running elbow analysis across K values...')
for K in K_values:
    km = MiniBatchKMeans(
        n_clusters=K, init='k-means++', n_init=3, batch_size=10000, random_state=42
    )
    km.fit(X_scaled_all)
    inertias.append(km.inertia_)
    print(f'  K={K:2d} | Inertia={km.inertia_:.2f}')

# Mathematical elbow deduction (second-order difference)
inertias_arr = np.array(inertias)
delta2 = np.diff(np.diff(inertias_arr))
elbow_idx = np.argmin(delta2) + 1  # +1 offset
K_elbow = K_values[elbow_idx]

K_CHOSEN = 30

plt.figure(figsize=(12, 5))
plt.plot(K_values, inertias, 'bo-', linewidth=2, markersize=7, label='Inertia')
plt.axvline(x=K_elbow, color='red', linestyle='--', label=f'Mathematical Elbow: K={K_elbow}')
plt.axvline(x=K_CHOSEN, color='green', linestyle=':', label=f'Chosen K={K_CHOSEN}')
plt.xlabel('Number of Clusters K')
plt.ylabel('Inertia')
plt.title('Elbow Analysis Curve')
plt.legend()
plt.grid(True)
plt.savefig('elbow_plot.png', dpi=150)
plt.show()

---
## CELL 11: Train Final KMeans Cluster Model

Train the production-grade KMeans model with `n_init=10` and save outputs.

In [ ]:
# CELL 11: KMeans Training & Target Extraction
# =============================================================================

print(f"🎯 Training final KMeans Clusterer (K={K_CHOSEN}, 10 restarts)... ")
km_final = KMeans(
    n_clusters=K_CHOSEN,
    init='k-means++', 
    n_init=10, 
    max_iter=500, 
    tol=1e-5,
    random_state=42
)

km_final.fit(X_scaled_all)

cluster_labels = km_final.labels_
centroids = km_final.cluster_centers_

print("✅ Final KMeans Cluster Model trained successfully.")
print(f"  - Inertia: {km_final.inertia_:.2f}")
print(f"  - Centroids array: {centroids.shape}")

# Size analysis
sizes = pd.Series(cluster_labels).value_counts().sort_index()
print(f"\nCluster Size demographics:")
print(f"  - Min: {sizes.min()}")
print(f"  - Max: {sizes.max()}")
print(f"  - Mean: {sizes.mean():.0f}")
print(f"  - Std Dev: {sizes.std():.0f}")

# Save centroids and model
joblib.dump(km_final, 'cluster_model.pkl')
np.save('centroids.npy', centroids)
np.save('cluster_assignments.npy', cluster_labels)
np.save('cluster_feature_names.npy', np.array(feature_names))
print("💾 Unsupervised cluster models saved locally.")

---
## CELL 12: l-Diversity Computation

Deduce class diversity in each cluster to assert traffic obfuscation and confusion capability.

In [ ]:
# CELL 12: l-Diversity evaluation
# =============================================================================

cluster_info = defaultdict(lambda: {'size': 0, 'Benign': 0, 'Malicious': 0})

for sample_cluster, sample_label in zip(cluster_labels, y_raw):
    cluster_info[sample_cluster]['size'] += 1
    lbl_str = str(sample_label)
    if 'enign' in lbl_str or lbl_str == '0':
        cluster_info[sample_cluster]['Benign'] += 1
    else:
        cluster_info[sample_cluster]['Malicious'] += 1

l_diversity_per_cluster = {}
for cid, info in cluster_info.items():
    distinct_classes = sum(1 for k in ['Benign', 'Malicious'] if info[k] > 0)
    l_diversity_per_cluster[cid] = {
        'size':         info['size'],
        'benign_count': info['Benign'],
        'mal_count':    info['Malicious'],
        'l_diversity':  distinct_classes,
        'benign_pct':   info['Benign'] / info['size'] * 100,
        'mal_pct':      info['Malicious'] / info['size'] * 100
    }

l_div_df = pd.DataFrame(l_diversity_per_cluster).T
l_div_df.index.name = 'cluster_id'
l_div_df = l_div_df.sort_values('size', ascending=False)

print("=== l-DIVERSITY METRIC CHART ===")
print(l_div_df.to_string())

min_sz  = l_div_df['size'].min()
mean_sz = l_div_df['size'].mean()

l_div_df.to_csv('l_diversity_report.csv')
print("\n✅ Saved report as `l_diversity_report.csv`.")

---
## CELL 13: Mathematical Privacy Proof Calculation

Solve formal attacker accuracy constraints combining k-Anonymity and Differential Privacy:  
$$P_{\text{attack}} \le \frac{1}{k} + \exp(-\varepsilon)$$

In [ ]:
# CELL 13: Privacy Bound Resolution
# =============================================================================

epsilon_values = [0.5, 1.0, 2.0]

print('=== FORMAL PRIVACY BOUNDS ===')
print(f'Minimum cluster size (k): {min_sz}\n')
print(f'{"ε":>6} | {"1/k":>8} | {"exp(-ε)":>9} | {"P_attack ≤":>12} | {"Interpretation"}')
print('-' * 70)

for eps in epsilon_values:
    k_term   = 1.0 / min_sz
    dp_term  = math.exp(-eps)
    bound    = k_term + dp_term
    pct      = bound * 100
    
    if eps == 0.5:
        interp = "High privacy, higher timing overhead"
    elif eps == 1.0:
        interp = "Balanced (DoH-Shield standard prototype config)"
    else:
        interp = "Lower privacy, minimal overhead"
    
    print(f'{eps:>6.1f} | {k_term:>8.6f} | {dp_term:>9.4f} | {bound:>10.4f} ({pct:>4.1f}%) | {interp}')

print(f'\n📄 Upper bound attacker success rate using standard ε=1.0: {(1/min_sz + math.exp(-1.0))*100:.2f}%')

---
## CELL 14: Visualise Clusters and Profile Centroids

Draw PCA-projected cluster divisions, size distribution histograms, and top-feature heatmaps that directly feed into the paper.

In [ ]:
# CELL 14: Cluster Profiles Mapping
# =============================================================================

viz_idx  = np.random.choice(len(X_scaled_all), size=30000, replace=False)
X_viz    = X_scaled_all[viz_idx]
labels_viz = cluster_labels[viz_idx]

pca_full = PCA(n_components=2, random_state=42)
X_viz_2d = pca_full.fit_transform(X_viz)
centroids_2d = pca_full.transform(centroids)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Scatter PCA
axes[0].scatter(X_viz_2d[:, 0], X_viz_2d[:, 1], c=labels_viz, cmap='tab20', s=3, alpha=0.3)
axes[0].scatter(centroids_2d[:, 0], centroids_2d[:, 1], c='black', s=120, marker='X', zorder=5, label='Centroids')
for i, (cx, cy) in enumerate(centroids_2d):
    axes[0].annotate(str(i), (cx, cy), fontsize=7, ha='center', color='white', fontweight='bold',
                     bbox=dict(boxstyle='round,pad=0.2', facecolor='black', alpha=0.7))
axes[0].set_title('K-Means Cluster Space (PCA Projection)')
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')
axes[0].legend()

# Demographics bar
sizes_sorted = l_div_df['size'].values
axes[1].bar(range(K_CHOSEN), sorted(sizes_sorted, reverse=True), color='steelblue', alpha=0.8)
axes[1].axhline(y=min_sz, color='red', linestyle='--', label=f'Min size = {min_sz}')
axes[1].axhline(y=mean_sz, color='green', linestyle='-.', label=f'Mean size = {mean_sz:.0f}')
axes[1].set_xlabel('Sorted Clusters')
axes[1].set_ylabel('Count')
axes[1].set_title('Cluster Size demographics')
axes[1].legend()

plt.tight_layout()
plt.savefig('cluster_visualization.png', dpi=150)
plt.show()

# Profile Centroid Heatmap for Top Features
top_features_list = np.load('top_features_phase3.npy', allow_pickle=True).tolist()
feature_names_arr = np.array(feature_names)
top_feat_indices = [list(feature_names_arr).index(f) for f in top_features_list if f in feature_names_arr]

centroid_top = centroids[:, top_feat_indices]

plt.figure(figsize=(14, 8))
sns.heatmap(
    centroid_top.T, xticklabels=[f'C{i}' for i in range(K_CHOSEN)],
    yticklabels=[top_features_list[i] if i < len(top_features_list) else f'F{i}' for i in range(len(top_feat_indices))],
    cmap='RdBu_r', center=0, annot=False, linewidths=0.3
)
plt.title('Cluster Centroid Target Features (Top 10 Attacker Leakage Profiles)')
plt.xlabel('Cluster Target Profile ID')
plt.ylabel('Leakage Dimension')
plt.tight_layout()
plt.savefig('centroid_heatmap.png', dpi=150)
plt.show()

---
## CELL 15: Compression of Artifacts

Compress all model coefficients, Standard scalers, metrics reports, and paper figures into a single zipped archive for simple download.

In [ ]:
# CELL 15: Create Zip Archive
# =============================================================================

artifacts = [
    'rf_attack_model.pkl', 'df_attack_model_best.pt', 'feature_scaler.pkl', 'label_encoder.pkl',
    'feature_names.npy', 'top_features_phase3.npy', 'cluster_model.pkl', 'cluster_scaler.pkl',
    'centroids.npy', 'cluster_assignments.npy', 'cluster_feature_names.npy', 'l_diversity_report.csv',
    'pca_cluster_structure.png', 'elbow_plot.png', 'cluster_visualization.png', 'centroid_heatmap.png',
    'eda_feature_distributions.png', 'eda_correlation_heatmap.png', 'rf_attack_results.png',
    'rf_feature_importance.png', 'cnn_training_curves.png'
]

zip_name = 'doh_shield_artifacts.zip'
print(f"📦 Generating compressed zip package: {zip_name}...")

with zipfile.ZipFile(zip_name, 'w') as zipf:
    for f in artifacts:
        if os.path.exists(f):
            zipf.write(f)
            print(f"  -> Compressed {f}")
        else:
            print(f"  ⚠️ File not found: {f}")

print(f"\n✅ Zipping process completed! Download the file: `{zip_name}` from Colab's file tree.")

---
## CELL 16: Optional Hugging Face Hub Upload

If `HF_token` is saved in Colab secrets, this cell logs in programmatically and uploads the completed zip containing all models to Hugging Face.

In [ ]:
# CELL 16: Upload zipped models to Hugging Face Repository
# =============================================================================
from huggingface_hub import HfApi, login

try:
    hf_token = userdata.get('HF_token')
    if hf_token:
        print("✅ Hugging Face authentication token found in Colab Secrets.")
        login(token=hf_token)
        
        api = HfApi()
        
        # Ask user for repo selection
        repo_name = input("Enter target Hugging Face repo ID (e.g. username/doh-shield-weights) or press Enter to skip: ").strip()
        if repo_name:
            print(f"📤 Creating/Locating Hugging Face Repository: {repo_name}...")
            api.create_repo(repo_id=repo_name, exist_ok=True, private=True)
            
            print(f"📤 Uploading zip bundle `{zip_name}`...")
            api.upload_file(
                path_or_fileobj=zip_name,
                path_in_repo=zip_name,
                repo_id=repo_name
            )
            print(f"✅ Successfully uploaded models to: https://huggingface.co/{repo_name}")
        else:
            print("⏭️ Upload skipped by user.")
    else:
        print("ℹ️ No Hugging Face token found in Colab secrets. Skipping optional Hub upload.")
except Exception as e:
    print(f"⚠️ Hugging Face Upload failed: {e}")
    print("You can download the zip locally and upload manually to Hugging Face if desired.")